In [ ]:
!pip install byaldi together pdf2image

In [ ]:
!sudo apt-get update
!sudo apt-get install -y poppler-utils

### Initialize the ColPali Model

In [ ]:
!pip install -U "torchao>=0.16.0"

In [ ]:
import os
from pathlib import Path
from byaldi import RAGMultiModalModel

# Initialize RAGMultiModalModel
model = RAGMultiModalModel.from_pretrained("vidore/colqwen2-v0.1")

In [ ]:
!pip install gradio -q

In [ ]:
!pip install filetype -q
!pip install python-docx -q
!pip install python-pptx -q

In [ ]:
!pip install reportlab -q

# **MultiModal RAG with ColQwen2 + Llama3.2 vision**

In [ ]:
import os
import base64
from pathlib import Path
from byaldi import RAGMultiModalModel
from together import Together
import gradio as gr
from PIL import Image
import io
from docx import Document
from pptx import Presentation
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import tempfile

# Helper function to convert DOCX to text
def docx_to_text(docx_path):
    doc = Document(docx_path)
    text = []
    for paragraph in doc.paragraphs:
        text.append(paragraph.text)
    return "\n".join(text)

# Helper function to convert PPTX to text
def pptx_to_text(pptx_path):
    presentation = Presentation(pptx_path)
    text = []
    for slide in presentation.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                text.append(shape.text)
    return "\n".join(text)

# Helper function to convert TXT to text
def txt_to_text(txt_path):
    with open(txt_path, 'r', encoding='utf-8') as file:
        return file.read()

# Helper function to convert text to PDF
def text_to_pdf(text, pdf_path):
    c = canvas.Canvas(pdf_path, pagesize=letter)
    width, height = letter
    lines = text.split('\n')
    y = height - 40
    for line in lines:
        if y < 40:
            c.showPage()
            y = height - 40
        c.drawString(40, y, line)
        y -= 12
    c.save()

# Function to handle MultiModal RAG process
def multimodal_rag(api_key, uploaded_file, user_query):
    try:
        # Set the Together AI API Key
        os.environ["TOGETHER_API_KEY"] = api_key
        client = Together(api_key=api_key)

        # Save the uploaded file to a local path
        file_path = uploaded_file.name
        file_extension = Path(file_path).suffix.lower()

        # Initialize ColPali Model
        model = RAGMultiModalModel.from_pretrained("vidore/colqwen2-v0.1")

        # Handle different file types
        if file_extension == ".pdf":
            input_path = Path(file_path)
        elif file_extension == ".docx":
            text = docx_to_text(file_path)
            temp_pdf_path = tempfile.mkstemp(suffix=".pdf")[1]
            text_to_pdf(text, temp_pdf_path)
            input_path = Path(temp_pdf_path)
        elif file_extension in [".ppt", ".pptx"]:
            text = pptx_to_text(file_path)
            temp_pdf_path = tempfile.mkstemp(suffix=".pdf")[1]
            text_to_pdf(text, temp_pdf_path)
            input_path = Path(temp_pdf_path)
        elif file_extension == ".txt":
            text = txt_to_text(file_path)
            temp_pdf_path = tempfile.mkstemp(suffix=".pdf")[1]
            text_to_pdf(text, temp_pdf_path)
            input_path = Path(temp_pdf_path)
        elif file_extension in [".png", ".jpg", ".jpeg"]:
            input_path = Path(file_path)
        else:
            return f"Unsupported file type: {file_extension}", None

        # Index the uploaded document
        index_name = "user_uploaded_index"
        model.index(
            input_path=input_path,
            index_name=index_name,
            store_collection_with_index=True,
            overwrite=True,
        )

        # Query the indexed document
        results = model.search(user_query, k=1)
        if not results:
            return "No relevant pages found in the document.", None

        retrieved_page = results[0].base64

        # Decode base64 to display the image
        image_data = base64.b64decode(retrieved_page)
        image = Image.open(io.BytesIO(image_data))

        # Query the Together AI API
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.2-90B-Vision-Instruct-Turbo",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": user_query},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{retrieved_page}"}}  # Assuming the image is a jpeg
                    ],
                }
            ],
            max_tokens=300,
        )

        # Return the response and the retrieved page image
        answer = response.choices[0].message.content
        return answer, image

    except Exception as e:
        return f"An error occurred: {str(e)}", None

# Build the Gradio Interface
with gr.Blocks() as app:
    gr.Markdown("# MultiModal RAG with ColQwen2 and Llama 3.2 Vision")
    gr.Markdown("Upload a document, provide your Together AI API key, and ask a question about the document.")

    api_key_input = gr.Textbox(label="Enter your Together AI API Key", placeholder="Paste your API key here", type="password")
    file_upload = gr.File(label="Upload Document", file_types=[".pdf", ".ppt", ".pptx", ".doc", ".docx", ".png", ".jpg", ".jpeg", ".txt"])
    user_query = gr.Textbox(label="Enter Your Question", placeholder="Type your question about the document here")
    response_output = gr.Textbox(label="Answer", placeholder="The response will appear here.", lines=5)
    image_output = gr.Image(label="Retrieved Page Image")

    submit_button = gr.Button("Submit")
    submit_button.click(multimodal_rag, inputs=[api_key_input, file_upload, user_query], outputs=[response_output, image_output])

app.launch()


# **MultiModal RAG with ColQwen2 + GPT-4o**

In [ ]:
import os
import base64
from pathlib import Path
import io
import tempfile

import gradio as gr
from PIL import Image
from byaldi import RAGMultiModalModel
from openai import OpenAI

from docx import Document
from pptx import Presentation
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


# =========================
# File conversion helpers
# =========================

def docx_to_text(docx_path):
    doc = Document(docx_path)
    return "\n".join(p.text for p in doc.paragraphs)


def pptx_to_text(pptx_path):
    presentation = Presentation(pptx_path)
    text = []
    for slide in presentation.slides:
        for shape in slide.shapes:
            if hasattr(shape, "text"):
                text.append(shape.text)
    return "\n".join(text)


def txt_to_text(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        return f.read()


def text_to_pdf(text, pdf_path):
    c = canvas.Canvas(pdf_path, pagesize=letter)
    width, height = letter
    y = height - 40

    for line in text.split("\n"):
        if y < 40:
            c.showPage()
            y = height - 40
        c.drawString(40, y, line)
        y -= 12

    c.save()


# =========================
# Multimodal RAG function
# =========================

def multimodal_rag(openai_api_key, uploaded_file, user_query):
    try:
        # ---- OpenAI client ----
        os.environ["OPENAI_API_KEY"] = openai_api_key
        client = OpenAI()

        # ---- Prepare file ----
        file_path = uploaded_file.name
        ext = Path(file_path).suffix.lower()

        if ext == ".pdf":
            input_path = Path(file_path)

        elif ext == ".docx":
            text = docx_to_text(file_path)
            temp_pdf = tempfile.mkstemp(suffix=".pdf")[1]
            text_to_pdf(text, temp_pdf)
            input_path = Path(temp_pdf)

        elif ext in [".ppt", ".pptx"]:
            text = pptx_to_text(file_path)
            temp_pdf = tempfile.mkstemp(suffix=".pdf")[1]
            text_to_pdf(text, temp_pdf)
            input_path = Path(temp_pdf)

        elif ext == ".txt":
            text = txt_to_text(file_path)
            temp_pdf = tempfile.mkstemp(suffix=".pdf")[1]
            text_to_pdf(text, temp_pdf)
            input_path = Path(temp_pdf)

        elif ext in [".png", ".jpg", ".jpeg"]:
            input_path = Path(file_path)

        else:
            return f"Unsupported file type: {ext}", None

        # ---- Load ColQwen2 ----
        model = RAGMultiModalModel.from_pretrained(
            "vidore/colqwen2-v0.1"
        )

        # ---- Index document ----
        model.index(
            input_path=input_path,
            index_name="user_uploaded_index",
            store_collection_with_index=True,
            overwrite=True,
        )

        # ---- Retrieve relevant page ----
        results = model.search(user_query, k=1)

        if not results:
            return "No relevant page found.", None

        retrieved_page_base64 = results[0].base64

        # ---- Decode image for UI ----
        image_bytes = base64.b64decode(retrieved_page_base64)
        image = Image.open(io.BytesIO(image_bytes))

        # ---- GPT-4o Vision Reasoning ----
        response = client.responses.create(
            model="gpt-4o",
            input=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "input_text",
                            "text": user_query
                        },
                        {
                            "type": "input_image",
                            "image_url": f"data:image/jpeg;base64,{retrieved_page_base64}"
                        }
                    ]
                }
            ],
            max_output_tokens=300
        )


        answer = response.output_text

        return answer, image

    except Exception as e:
        return f"Error: {str(e)}", None


# =========================
# Gradio UI
# =========================

with gr.Blocks() as app:
    gr.Markdown("# MultiModal RAG with ColQwen2 + GPT-4o")
    gr.Markdown("Upload a document and ask questions grounded in the content.")

    api_key_input = gr.Textbox(
        label="OpenAI API Key",
        type="password",
        placeholder="sk-..."
    )

    file_upload = gr.File(
        label="Upload Document",
        file_types=[".pdf", ".docx", ".pptx", ".txt", ".png", ".jpg", ".jpeg"]
    )

    user_query = gr.Textbox(
        label="Your Question",
        placeholder="Ask something about the document..."
    )

    answer_output = gr.Textbox(
        label="Answer",
        lines=6
    )

    image_output = gr.Image(
        label="Retrieved Page"
    )

    submit = gr.Button("Submit")

    submit.click(
        multimodal_rag,
        inputs=[api_key_input, file_upload, user_query],
        outputs=[answer_output, image_output]
    )

app.launch()
